# Notebook 05 - Synthetic Data Generation

Generates diverse training examples for all 10 cognitive distortion labels using
template-based slot filling. Each template has multiple slots with varied values
covering different contexts (work, social, academic, relationships, self-esteem).

**Target:** ~300 examples per distortion → ~3000 total synthetic examples  
**Output:** `data/synthetic/synthetic_data.csv`  
**Run:** Locally - no GPU needed.

These examples are merged with the original 2020 training examples before RoBERTa fine-tuning.

In [1]:
import random
import pandas as pd
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parent))
from src.config import DISTORTION_LABELS, DATA_SYN_DIR

random.seed(42)
os.makedirs(DATA_SYN_DIR, exist_ok=True)

print(f'Distortions: {len(DISTORTION_LABELS)}')
print(f'Output dir: {DATA_SYN_DIR}')

Distortions: 10
Output dir: D:\Projects\NLP_CBT\data\synthetic


## Template Definitions

Each distortion has single-sentence and multi-sentence templates.
Slots (e.g. `{context}`, `{outcome}`) are filled randomly from defined value lists.

In [2]:
# ── Shared slot values ─────────────────────────────────────────────────────────
CONTEXTS = {
    'work':         ['presentation', 'project', 'meeting', 'interview', 'report', 'task', 'deadline'],
    'academic':     ['exam', 'assignment', 'essay', 'test', 'class', 'lecture', 'dissertation'],
    'social':       ['party', 'conversation', 'gathering', 'date', 'hangout', 'event', 'dinner'],
    'relationship': ['argument', 'conversation', 'moment', 'situation', 'misunderstanding'],
    'general':      ['thing', 'situation', 'opportunity', 'chance', 'moment', 'attempt'],
}

ALL_CONTEXTS = [v for vals in CONTEXTS.values() for v in vals]

FREQ_WORDS   = ['always', 'never', 'constantly', 'perpetually', 'inevitably', 'every single time']
PEOPLE       = ['they', 'he', 'she', 'everyone', 'my boss', 'my friends', 'my colleagues', 'people']
THINK_VERBS  = ['think', 'believe', 'know', 'feel', 'assume', 'suspect']
NEG_LABELS   = ['a failure', 'useless', 'incompetent', 'worthless', 'an idiot', 'hopeless', 'stupid', 'a disappointment']
INTENSIFIERS = ['completely', 'totally', 'utterly', 'absolutely', 'entirely']

print('Slot values loaded.')

Slot values loaded.


In [3]:
# ── Template bank ──────────────────────────────────────────────────────────────
# Each distortion: list of (template_string, slots_dict)
# Slots are filled with random.choice() from the list.

TEMPLATES = {

# ── 1. All-or-nothing thinking ────────────────────────────────────────────────
'All-or-nothing thinking': [
    # single sentence
    ('Either I do this {qual} or it is a {total} failure.', {'qual': ['perfectly', 'flawlessly', 'exactly right', 'without any mistakes'], 'total': ['complete', 'total', 'utter', 'absolute']}),
    ('If it is not {perfect}, then it is {worthless}.', {'perfect': ['perfect', 'flawless', 'exactly right', 'done properly'], 'worthless': ['worthless', 'pointless', 'a waste', 'not good enough', 'meaningless']}),
    ('There is no middle ground - I either {succeed} or I {fail} {intensifier}.', {'succeed': ['succeed', 'win', 'nail it', 'get it right'], 'fail': ['fail', 'lose', 'mess it up', 'get it wrong'], 'intensifier': ['completely', 'totally', 'utterly']}),
    ('My {context} was either a {success} or a {disaster} - nothing in between.', {'context': ALL_CONTEXTS, 'success': ['success', 'triumph', 'win'], 'disaster': ['disaster', 'failure', 'catastrophe']}),
    ('I am either great at {context} or I am terrible. There is no in-between.', {'context': ALL_CONTEXTS}),
    ('If I cannot do this {right}, then there is no point doing it at all.', {'right': ['right', 'perfectly', 'correctly', 'properly', 'flawlessly']}),
    ('I got one thing wrong and that makes the whole {context} a {total} failure.', {'context': ALL_CONTEXTS, 'total': ['complete', 'total', 'utter']}),
    ('I see things as all good or all bad. There is never anything in the middle.', {}),
    ('Unless I do this {qual}, it counts for nothing.', {'qual': ['perfectly', 'flawlessly', 'without any mistakes', 'exactly right']}),
    ('My performance was either {good} or {bad} - I cannot see it any other way.', {'good': ['excellent', 'perfect', 'great'], 'bad': ['terrible', 'awful', 'a failure']}),
    # multi sentence
    ('I {intensifier} failed at my {context}. There is simply no middle ground for me. Either I nail it perfectly or it counts for nothing.', {'intensifier': ['completely', 'totally', 'utterly'], 'context': ALL_CONTEXTS}),
    ('If I do not do this perfectly, it is a total failure. I cannot see it any other way. Things are either perfect or they are worthless.', {}),
    ('My {context} did not go the way I wanted. As far as I am concerned, that makes it a complete disaster. I cannot see any in-between.', {'context': ALL_CONTEXTS}),
    ('I am either excellent at something or I am terrible. That is just how I think. There is never any grey area.', {}),
    ('Either this goes perfectly or it is not worth doing. I told myself that from the start. Now that it did not go perfectly, I feel like it was a complete waste.', {}),
],

# ── 2. Overgeneralization ─────────────────────────────────────────────────────
'Overgeneralization': [
    ('I {freq} mess up {context}.', {'freq': FREQ_WORDS, 'context': ALL_CONTEXTS}),
    ('Every time I try to do {context}, I fail.', {'context': ALL_CONTEXTS}),
    ('This {freq} happens to me. I cannot catch a break.', {'freq': ['always', 'constantly', 'inevitably', 'perpetually']}),
    ('I {freq} disappoint people no matter how hard I try.', {'freq': FREQ_WORDS}),
    ('Nothing {freq} goes right for me.', {'freq': ['ever', 'always', 'ever seems to']}),
    ('Whenever I attempt something like this, it {outcome}.', {'outcome': ['falls apart', 'goes wrong', 'fails', 'ends badly', 'blows up']}),
    ('I never get things right. It happens every single time.', {}),
    ('Everyone around me seems to succeed. I {freq} fall short.', {'freq': ['always', 'constantly', 'inevitably', 'perpetually']}),
    ('Things {freq} work out for other people but never for me.', {'freq': ['always', 'somehow always']}),
    ('I have tried this before and it {freq} ends the same way - badly.', {'freq': ['always', 'invariably', 'inevitably', 'constantly']}),
    # multi sentence
    ('I always mess up {context}. Every single time I try, something goes wrong. It is a pattern I cannot escape.', {'context': ALL_CONTEXTS}),
    ('This always happens to me. No matter what I do or how hard I try, things fall apart. It has been this way my whole life.', {}),
    ('I never get it right. Every time I think things might be different, they end up the same. I always end up failing.', {}),
    ('Every time I try to do {context}, it goes wrong. I am starting to think I am just not capable. This always happens.', {'context': ALL_CONTEXTS}),
    ('Nothing ever works out for me. I have tried so many times and it always ends badly. I can never seem to get ahead.', {}),
],

# ── 3. Mental filter ─────────────────────────────────────────────────────────
'Mental filter': [
    ('Even though {pos}, all I can think about is the {neg}.', {'pos': ['most of it went well', 'I got good feedback', 'people said nice things', 'a lot was positive'], 'neg': ['one mistake', 'single criticism', 'one negative comment', 'one thing that went wrong']}),
    ('I got {pos} feedback on my {context} but I cannot stop focusing on the {neg} comment.', {'pos': ['mostly positive', 'a lot of good', 'great', 'really good'], 'context': ALL_CONTEXTS, 'neg': ['one negative', 'single critical', 'one bad']}),
    ('Everything else went fine but that one {neg} is all I can think about.', {'neg': ['mistake', 'error', 'criticism', 'negative remark', 'thing that went wrong']}),
    ('The whole {context} went well except for one moment, and that is the only thing I remember.', {'context': ALL_CONTEXTS}),
    ('I keep replaying the {neg} over and over in my head and ignoring everything else.', {'neg': ['one mistake', 'single error', 'bad moment', 'one thing that went wrong', 'negative comment']}),
    ('People said ten good things and one bad thing and I can only focus on the bad thing.', {}),
    ('No matter how many things go right, one thing going wrong ruins the whole experience for me.', {}),
    ('I am filtering out all the positives and fixating on that one negative thing that happened.', {}),
    ('My {context} had many good parts but I am obsessing over the single part that did not go well.', {'context': ALL_CONTEXTS}),
    ('I received a lot of praise but one piece of criticism is consuming my entire attention.', {}),
    # multi sentence
    ('My {context} went really well overall. But there was one moment where I stumbled, and I cannot get it out of my head. Everything else is a blur.', {'context': ALL_CONTEXTS}),
    ('Three people complimented my work and one person had a small criticism. I have spent the whole day thinking about the criticism and barely registered the compliments.', {}),
    ('Most of the feedback was positive. But I zeroed in on the one negative comment and I have been replaying it ever since. I cannot seem to filter it out.', {}),
    ('The {context} had a lot of highlights. But one small thing went wrong and it has completely overshadowed everything good that happened.', {'context': ALL_CONTEXTS}),
    ('Everything was going well until one mistake happened. Now that is all I can think about. I know I am ignoring all the good parts but I cannot help it.', {}),
],

# ── 4. Should statements ─────────────────────────────────────────────────────
'Should statements': [
    ('I should {action} by now.', {'action': ['know better', 'be further along', 'have this figured out', 'be doing better', 'be more successful', 'be over this']}),
    ('I must {action} or I am failing.', {'action': ['work harder', 'do better', 'be more productive', 'try harder', 'achieve more']}),
    ('I should not {feeling} like this.', {'feeling': ['feel', 'be thinking', 'be reacting', 'be struggling', 'be upset']}),
    ('People ought to {action} more.', {'action': ['understand', 'care', 'try', 'be more considerate', 'appreciate what I do', 'respect my time']}),
    ('I should have {past_action} better.', {'past_action': ['done', 'prepared', 'handled it', 'responded', 'managed it', 'known']}),
    ('I must not make {mistakes} if I want to succeed.', {'mistakes': ['mistakes', 'errors', 'any errors', 'any mistakes', 'these kinds of errors']}),
    ('I ought to be able to handle this without {struggle}.', {'struggle': ['struggling', 'feeling overwhelmed', 'needing help', 'feeling anxious', 'feeling stressed']}),
    ('I should be more {quality} than I am.', {'quality': ['disciplined', 'organised', 'productive', 'confident', 'resilient', 'capable']}),
    ('I must always {action} no matter how I feel.', {'action': ['be positive', 'perform at my best', 'keep it together', 'stay calm', 'push through']}),
    ('I should never {action} in a professional setting.', {'action': ['show weakness', 'ask for help', 'admit I do not know', 'make mistakes', 'show emotion']}),
    # multi sentence
    ('I should have handled that {context} better. I knew what to do and I still did not do it properly. I should be better than this by now.', {'context': ALL_CONTEXTS}),
    ('I must not feel anxious about this. Strong people do not get anxious. I should be able to push through without feeling like this.', {}),
    ('I should be further along in my career by now. Other people my age have achieved so much more. I ought to have worked harder.', {}),
    ('People should be more understanding. They ought to know that I am doing my best. They must realise that I cannot do everything perfectly.', {}),
    ('I should not need help with this. I must be able to figure it out on my own. Needing support is something I should have grown out of by now.', {}),
],

# ── 5. Labeling ───────────────────────────────────────────────────────────────
'Labeling': [
    ('I am {label}.', {'label': NEG_LABELS}),
    ('I am just {label}. That is all there is to it.', {'label': NEG_LABELS}),
    ('I made a mistake and that proves I am {label}.', {'label': NEG_LABELS}),
    ('She is {label} for doing that.', {'label': ['completely selfish', 'a terrible person', 'so inconsiderate', 'a bad friend', 'truly awful']}),
    ('He is {label}. That is just who he is.', {'label': ['a loser', 'an idiot', 'completely useless', 'hopeless', 'a disappointment']}),
    ('I failed at my {context} so I must be {label}.', {'context': ALL_CONTEXTS, 'label': NEG_LABELS}),
    ('I am fundamentally {label} and I cannot change that.', {'label': ['broken', 'flawed', 'incapable', 'not good enough', 'a failure']}),
    ('The fact that I struggled with {context} just confirms that I am {label}.', {'context': ALL_CONTEXTS, 'label': NEG_LABELS}),
    ('I am {label} and everyone can see it.', {'label': NEG_LABELS}),
    ('I am not just someone who makes mistakes - I am {label}.', {'label': NEG_LABELS}),
    # multi sentence
    ('I made a mistake at my {context}. I am a failure. There is no other way to describe it.', {'context': ALL_CONTEXTS}),
    ('I struggled through that {context}. It just confirms what I already knew - I am {label}. That is who I am.', {'context': ALL_CONTEXTS, 'label': NEG_LABELS}),
    ('She did that to me on purpose. She is a terrible person. There is nothing more to say about it.', {}),
    ('I cannot do this. I have tried and tried and I keep failing. I am just {label}. That is the truth.', {'label': NEG_LABELS}),
    ('He embarrassed me in front of everyone. He is selfish and inconsiderate. That is just who he is as a person.', {}),
],

# ── 6. Personalization ────────────────────────────────────────────────────────
'Personalization': [
    ('It is {blame} fault that this happened.', {'blame': ['all my', 'entirely my', 'completely my', 'mostly my']}),
    ('If only I had {past_action}, none of this would have happened.', {'past_action': ['done things differently', 'been more careful', 'paid more attention', 'been better prepared', 'acted sooner']}),
    ('Their {reaction} is because of something I did.', {'reaction': ['mood', 'reaction', 'silence', 'anger', 'sadness', 'frustration', 'bad day']}),
    ('I am responsible for how this turned out even though I could not control everything.', {}),
    ('The failure of the {context} is on me.', {'context': ['project', 'team', 'relationship', 'situation', 'group effort', 'collaboration']}),
    ('I must have done something to make {person} feel this way.', {'person': ['them', 'her', 'him', 'everyone', 'my colleague', 'my partner', 'my friend']}),
    ('Whenever something goes wrong around me, I assume it is because of me.', {}),
    ('{Person} is upset and I am sure it is because of something I said or did.', {'Person': ['She', 'He', 'My manager', 'My friend', 'My partner', 'Everyone']}),
    ('I take responsibility for things that are not even fully in my control.', {}),
    ('The whole thing fell apart and I keep thinking it was all my doing.', {}),
    # multi sentence
    ('The {context} did not go well. I keep blaming myself even though there were so many factors involved. I feel like it is all my fault.', {'context': ALL_CONTEXTS}),
    ('She seemed upset today. I immediately assumed I had done something wrong. I cannot help feeling responsible for her mood.', {}),
    ('The team project failed. I know others contributed to the problems but I feel like it is primarily my fault. I keep going over what I could have done differently.', {}),
    ('He did not respond to my message. I spent all day thinking about what I said wrong. It must be something I did.', {}),
    ('If only I had handled things differently, none of this would have happened. I take on all the blame even when things are out of my control.', {}),
],

# ── 7. Catastrophising/Magnification ─────────────────────────────────────────
'Catastrophising/Magnification': [
    ('This is a {disaster} and I will never recover from it.', {'disaster': ['complete disaster', 'catastrophe', 'total failure', 'nightmare']}),
    ('If this {context} goes wrong, my {life_part} is over.', {'context': ALL_CONTEXTS, 'life_part': ['career', 'future', 'life', 'reputation', 'relationship']}),
    ('Everything is {ruined} because of what happened.', {'ruined': ['ruined', 'falling apart', 'destroyed', 'over', 'finished']}),
    ('This {small_thing} is going to have {consequence}.', {'small_thing': ['small mistake', 'minor setback', 'one error', 'single failure'], 'consequence': ['devastating consequences', 'terrible outcomes', 'catastrophic results', 'irreversible damage']}),
    ('I cannot come back from this. It is too {bad}.', {'bad': ['bad', 'serious', 'catastrophic', 'devastating', 'terrible']}),
    ('One bad {context} and now everything I have worked for is {gone}.', {'context': ALL_CONTEXTS, 'gone': ['gone', 'ruined', 'destroyed', 'wasted', 'finished']}),
    ('This is the worst thing that could possibly happen to me right now.', {}),
    ('What started as a small problem has become an absolute {disaster}.', {'disaster': ['disaster', 'catastrophe', 'nightmare', 'crisis']}),
    ('If people find out about this {context}, it will {consequence}.', {'context': ALL_CONTEXTS, 'consequence': ['ruin me', 'destroy everything', 'be catastrophic', 'end my career']}),
    ('This is so much worse than it seems. I can see exactly how bad this is going to get.', {}),
    # multi sentence
    ('I made a mistake at my {context}. This is a complete disaster. I will never recover from this and everything I have worked for is ruined.', {'context': ALL_CONTEXTS}),
    ('One bad result and now I am convinced everything is over. It feels catastrophic. I cannot see any way this does not end terribly.', {}),
    ('This {context} did not go well. It is the worst thing that could have happened. My career is finished and I do not know how to come back from this.', {'context': ALL_CONTEXTS}),
    ('A small problem turned into what feels like a total catastrophe. I know I am probably making it bigger than it is but I cannot see past how terrible this feels.', {}),
    ('Everything is falling apart. One thing went wrong and now it feels like the whole structure of my life is collapsing. I am not sure I can recover.', {}),
],

# ── 8. Emotional Reasoning ────────────────────────────────────────────────────
'Emotional Reasoning': [
    ('I feel {emotion} so I must {conclusion}.', {'emotion': ['stupid', 'worthless', 'like a failure', 'like an idiot', 'incompetent', 'useless'], 'conclusion': ['be stupid', 'be worthless', 'be a failure', 'be incompetent', 'be useless']}),
    ('Because I feel {emotion}, it must be true.', {'emotion': ['anxious', 'afraid', 'worried', 'embarrassed', 'ashamed', 'guilty', 'overwhelmed']}),
    ('I feel like something is wrong, so something {must} be wrong.', {'must': ['must', 'has to', 'definitely']}),
    ('My feelings {freq} tell me the truth about a situation.', {'freq': ['always', 'never lie to me', 'are always right', 'cannot be wrong']}),
    ('I feel like I am going to fail, so I {must} be going to fail.', {'must': ['must', 'probably will', 'definitely will']}),
    ('I can feel that they do not like me, so they {must} not like me.', {'must': ['must', 'definitely do', 'obviously do']}),
    ('This situation feels dangerous, so it {must} be dangerous even if nothing has actually gone wrong.', {'must': ['must', 'has to', 'clearly']}),
    ('I feel like a burden, therefore I {must} actually be a burden to others.', {'must': ['must', 'definitely am', 'clearly am']}),
    ('The anxiety I feel is proof that something bad is about to happen.', {}),
    ('I feel embarrassed about my {context} so my {context} must have been embarrassing.', {'context': ALL_CONTEXTS}),
    # multi sentence
    ('I feel completely stupid right now. That means I must actually be stupid. My feelings do not lie to me about these things.', {}),
    ('I woke up feeling like something was going to go wrong today. It felt so real. That feeling must mean that something bad is actually going to happen.', {}),
    ('I feel like my colleagues do not respect me. I cannot shake the feeling. If I feel it this strongly, it must be true.', {}),
    ('I feel worthless after what happened at my {context}. I know feelings are just feelings but I cannot help believing that if I feel this way, I must actually be worthless.', {'context': ALL_CONTEXTS}),
    ('The guilt I feel is overwhelming. That must mean I really did something terrible. I trust my feelings over logic in situations like this.', {}),
],

# ── 9. Mind Reading ───────────────────────────────────────────────────────────
'Mind Reading': [
    ('I know {person} {think_verb} I am {label}.', {'person': PEOPLE, 'think_verb': THINK_VERBS, 'label': NEG_LABELS}),
    ('{Person} {signal} because {person2} {think_verb} I am {label}.', {'Person': ['She', 'He', 'They'], 'signal': ['did not reply', 'looked away', 'went quiet', 'avoided me', 'seemed distant'], 'person2': ['she', 'he', 'they'], 'think_verb': THINK_VERBS, 'label': NEG_LABELS}),
    ('I can tell {person} {think_verb} {neg} about me.', {'person': ['they', 'she', 'he', 'everyone', 'my boss', 'my colleagues'], 'think_verb': ['think', 'feel', 'believe'], 'neg': ['negatively', 'badly', 'poorly', 'the worst']}),
    ('Everyone could see that I was {state} during the {context}.', {'state': ['struggling', 'nervous', 'lost', 'panicking', 'out of my depth', 'embarrassed'], 'context': ALL_CONTEXTS}),
    ('I know exactly what {person} is thinking and it is not good.', {'person': PEOPLE}),
    ('{Person} definitely thinks I am {label}. I just know it.', {'Person': ['She', 'He', 'My manager', 'My boss', 'My colleague'], 'label': NEG_LABELS}),
    ('I could tell from {person} expression that {person2} was judging me.', {'person': ['their', 'her', 'his', 'everyone\'s'], 'person2': ['they were', 'she was', 'he was', 'everyone was']}),
    ('{Person} has not spoken to me today. I know it is because {person2} is angry with me.', {'Person': ['She', 'He', 'My manager', 'My colleague', 'My friend'], 'person2': ['she', 'he', 'they']}),
    ('I know what my {person} really thinks about my work and it is not flattering.', {'person': ['boss', 'manager', 'colleague', 'teacher', 'professor', 'supervisor']}),
    ('They laughed and I immediately knew they were laughing at me.', {}),
    # multi sentence
    ('My {person} did not reply to my message. I know exactly why. They think I am {label} and they do not want to deal with me.', {'person': ['colleague', 'boss', 'friend', 'manager'], 'label': NEG_LABELS}),
    ('I could tell from the look on her face that she thinks I am incompetent. I know what she was thinking. She has never rated me.', {}),
    ('Nobody said anything after my {context}. I know they were all thinking the same thing - that I was terrible. I could feel the judgment in the room.', {'context': ALL_CONTEXTS}),
    ('He went quiet after what I said. I know he is upset with me and thinks I am {label}. I just know it even though he has not said anything.', {'label': NEG_LABELS}),
    ('I know my colleagues think I am not up to the job. I can tell from the way they act around me. They do not need to say it out loud.', {}),
],

# ── 10. Fortune-telling ───────────────────────────────────────────────────────
'Fortune-telling': [
    ('I know this {context} is going to {outcome}.', {'context': ALL_CONTEXTS, 'outcome': ['go badly', 'fail', 'end terribly', 'be a disaster', 'go wrong']}),
    ('I am definitely going to {outcome} at this {context}.', {'outcome': ['fail', 'mess it up', 'embarrass myself', 'disappoint everyone', 'get it wrong'], 'context': ALL_CONTEXTS}),
    ('There is no way this is going to {outcome} well.', {'outcome': ['turn out', 'end', 'work out', 'go', 'work']}),
    ('I can already see how this is going to end and it is not {good}.', {'good': ['good', 'going to be pretty', 'going to work out', 'promising']}),
    ('I just know I am going to {outcome} this {context}.', {'outcome': ['fail', 'mess up', 'ruin', 'bomb'], 'context': ALL_CONTEXTS}),
    ('This is pointless. I already know how it is going to go.', {}),
    ('I have a feeling this {context} is going to {outcome} before it even starts.', {'context': ALL_CONTEXTS, 'outcome': ['fail', 'go wrong', 'end badly', 'be a disaster']}),
    ('I can predict right now that this is not going to work out for me.', {}),
    ('The {context} has not even started yet and I already know I am going to {outcome} it.', {'context': ALL_CONTEXTS, 'outcome': ['fail', 'mess up', 'bomb', 'ruin']}),
    ('I will never succeed at this. I know it before I even try.', {}),
    # multi sentence
    ('I have my {context} coming up and I already know how it is going to go. I am going to fail. I can see it coming and there is nothing I can do about it.', {'context': ALL_CONTEXTS}),
    ('There is no point even trying. I know exactly how this will end and it will not be good. I can predict my own failure before it happens.', {}),
    ('Everyone keeps telling me to stay positive about this {context}. But I know it is going to go badly. I can feel it. Nothing is going to change that outcome.', {'context': ALL_CONTEXTS}),
    ('I am preparing for my {context} but I already know the result. I am going to fail. I have no doubt about it.', {'context': ALL_CONTEXTS}),
    ('I just know this is not going to work out. I cannot explain it but I can feel that it is going to end badly. That feeling is always right.', {}),
],

}

print(f'Templates defined for {len(TEMPLATES)} distortions.')
for label, tmpl_list in TEMPLATES.items():
    print(f'  {label}: {len(tmpl_list)} templates')

Templates defined for 10 distortions.
  All-or-nothing thinking: 15 templates
  Overgeneralization: 15 templates
  Mental filter: 15 templates
  Should statements: 15 templates
  Labeling: 15 templates
  Personalization: 15 templates
  Catastrophising/Magnification: 15 templates
  Emotional Reasoning: 15 templates
  Mind Reading: 15 templates
  Fortune-telling: 15 templates


## Generation Function

In [4]:
def fill_template(template: str, slots: dict) -> str:
    """Fill a template string by randomly sampling each slot value."""
    result = template
    for slot, values in slots.items():
        if isinstance(values, list) and len(values) > 0:
            result = result.replace(f'{{{slot}}}', random.choice(values))
    return result


def generate_for_distortion(label: str, templates: list, target: int = 300) -> list:
    """
    Generate `target` examples for a single distortion label.
    Cycles through templates, sampling slots randomly each pass to maximise diversity.
    """
    examples = []
    seen = set()
    max_attempts = target * 20
    attempts = 0

    while len(examples) < target and attempts < max_attempts:
        template_str, slots = random.choice(templates)
        text = fill_template(template_str, slots).strip()
        if text not in seen:
            seen.add(text)
            examples.append(text)
        attempts += 1

    return examples


print('Generation function ready.')

Generation function ready.


## Generate All Examples

In [5]:
TARGET_PER_DISTORTION = 300

rows = []

for label in DISTORTION_LABELS:
    if label not in TEMPLATES:
        print(f'WARNING: No templates for {label}')
        continue

    examples = generate_for_distortion(label, TEMPLATES[label], target=TARGET_PER_DISTORTION)

    for text in examples:
        row = {'text': text}
        for lbl in DISTORTION_LABELS:
            row[lbl] = 1 if lbl == label else 0
        rows.append(row)

    print(f'{label:45s} → {len(examples)} examples generated')

df = pd.DataFrame(rows)
print(f'\nTotal synthetic examples: {len(df)}')
print(df[DISTORTION_LABELS].sum().to_string())

All-or-nothing thinking                       → 300 examples generated
Overgeneralization                            → 274 examples generated
Mental filter                                 → 300 examples generated
Should statements                             → 87 examples generated
Labeling                                      → 300 examples generated
Personalization                               → 72 examples generated
Catastrophising/Magnification                 → 300 examples generated
Emotional Reasoning                           → 119 examples generated
Mind Reading                                  → 300 examples generated
Fortune-telling                               → 300 examples generated

Total synthetic examples: 2352
All-or-nothing thinking          300
Overgeneralization               274
Mental filter                    300
Should statements                 87
Labeling                         300
Personalization                   72
Catastrophising/Magnification    300
E

## Quality Check - Sample Examples

In [6]:
print('=== SAMPLE EXAMPLES PER DISTORTION ===\n')
for label in DISTORTION_LABELS:
    subset = df[df[label] == 1]['text'].sample(3, random_state=42).tolist()
    print(f'[{label}]')
    for s in subset:
        print(f'  • {s}')
    print()

=== SAMPLE EXAMPLES PER DISTORTION ===

[All-or-nothing thinking]
  • I got one thing wrong and that makes the whole task a complete failure.
  • I am either great at gathering or I am terrible. There is no in-between.
  • I got one thing wrong and that makes the whole party a total failure.

[Overgeneralization]
  • Every time I try to do interview, it goes wrong. I am starting to think I am just not capable. This always happens.
  • Every time I try to do date, it goes wrong. I am starting to think I am just not capable. This always happens.
  • I always mess up report.

[Mental filter]
  • I got a lot of good feedback on my report but I cannot stop focusing on the single critical comment.
  • I got mostly positive feedback on my conversation but I cannot stop focusing on the single critical comment.
  • I got mostly positive feedback on my deadline but I cannot stop focusing on the one negative comment.

[Should statements]
  • I should have handled that essay better. I knew what to

## Save to CSV

In [7]:
out_path = os.path.join(DATA_SYN_DIR, 'synthetic_data.csv')
df.to_csv(out_path, index=False)
print(f'Saved {len(df)} examples to: {out_path}')
print(f'\nColumns: {list(df.columns)}')
print(f'Shape: {df.shape}')

Saved 2352 examples to: D:\Projects\NLP_CBT\data\synthetic\synthetic_data.csv

Columns: ['text', 'All-or-nothing thinking', 'Overgeneralization', 'Mental filter', 'Should statements', 'Labeling', 'Personalization', 'Catastrophising/Magnification', 'Emotional Reasoning', 'Mind Reading', 'Fortune-telling']
Shape: (2352, 11)
